# AQL Agent Demo

A smolagents-based agent that explores the **Archive Query Log (AQL)** — 557 M search result pages,
2.2 B web archive captures, and 775+ search providers spanning 25 years.

The agent connects to the AQL API via **MCP (Model Context Protocol)**, which publishes all available
tools automatically — no manual tool definitions needed.


## 1. Setup

In [8]:
!pip install -Uq smolagents requests smolagents[mcp] arxiv-mcp-server ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 46.1 MB/s eta 0:00:00


## 2. Configuration

Credentials are read from **Colab Secrets** (the key icon in the left sidebar).

Set `BACKEND` to choose the LLM provider:
- **`webis`** (default) — Webis Chat API, model `qwen3-30b-a3b`
- **`blablador`** — Helmholtz Blablador, model `alias-fast`

In [2]:
import os
from google.colab import userdata

# Choose backend: "webis" or "blablador"
BACKEND = "webis"

if BACKEND == "blablador":
    if not userdata.get("BLABLADOR_KEY"):
        print("⚠️  BLABLADOR_KEY not set — add it in Colab Secrets")
    else:
        print("✓ Credentials loaded (blablador)")
        print(f"  Blablador URL: {userdata.get('BLABLADOR_OPENAI_BASE')}")
else:
    if not userdata.get("WEBIS_API_KEY"):
        print("⚠️  WEBIS_API_KEY not set — add it in Colab Secrets")
    else:
        print("✓ Credentials loaded (webis)")
        print(f"  Webis URL: {userdata.get('WEBIS_OPENAI_BASE_URL')}")

AQL_API_BASE = userdata.get("AQL_API_BASE") or "https://aql-api-preview.web.webis.de"
print(f"  AQL API:   {AQL_API_BASE}")

✓ Credentials loaded (webis)
  Webis URL: https://chat.web.webis.de/api
  AQL API:   https://aql-api-preview.web.webis.de


## 3. AQL API

- **API docs**: https://aql-api-preview.web.webis.de/docs
- **MCP endpoint**: https://aql-api-preview.web.webis.de/mcp/

In [3]:
import requests

resp = requests.get(f"{AQL_API_BASE}/health", timeout=5)
assert resp.status_code == 200, f"API not reachable at {AQL_API_BASE} — is the server running?"
print(f"✓ AQL API is up at {AQL_API_BASE}")

✓ AQL API is up at https://aql-api-preview.web.webis.de


## 4. Model Setup

Uses `OpenAIServerModel` from smolagents — compatible with any OpenAI-style API.
To swap provider, change `api_base` and `model_id`.

In [4]:
from smolagents import OpenAIServerModel

if BACKEND == "blablador":
    model = OpenAIServerModel(
        model_id="alias-fast",
        api_base=userdata.get("BLABLADOR_OPENAI_BASE") or "https://api.helmholtz-blablador.fz-juelich.de/v1/",
        api_key=userdata.get("BLABLADOR_KEY"),
    )
else:
    model = OpenAIServerModel(
        model_id="qwen3-30b-a3b",
        api_base=userdata.get("WEBIS_OPENAI_BASE_URL"),
        api_key=userdata.get("WEBIS_API_KEY"),
    )

print(f"Using model: {model.model_id} via {BACKEND}")

Using model: qwen3-30b-a3b via webis


---
## MCP Agent

The AQL API exposes a **Model Context Protocol (MCP)** endpoint at `/mcp`.
smolagents' `MCPClient` connects and **discovers all available tools automatically** —
the server owns and publishes its definitions; no manual tool registrations needed.

In [5]:
# @title Infrastructure — run once (schema compatibility patch) { display-mode: "form" }
import subprocess as _subprocess
import smolagents.models as _sm_models
from smolagents import MCPClient, ToolCallingAgent


# ── Colab subprocess fix ──────────────────────────────────────────────────────
# Colab replaces sys.stderr with a fake stream that has no file descriptor.
# When stdio MCP servers are spawned as subprocesses, Popen calls stderr.fileno()
# and crashes. Patch Popen.__init__ to silently redirect to /dev/null instead.

_orig_Popen_init = _subprocess.Popen.__init__

def _popen_safe(self, *args, **kwargs):
    stderr = kwargs.get("stderr")
    if stderr is not None and not isinstance(stderr, int):
        try:
            stderr.fileno()
        except Exception:
            kwargs["stderr"] = _subprocess.DEVNULL
    _orig_Popen_init(self, *args, **kwargs)

_subprocess.Popen.__init__ = _popen_safe


# ── Schema normalisation ──────────────────────────────────────────────────────

def _clean_schema(s):
    if not isinstance(s, dict):
        return s
    if "anyOf" in s:
        non_null = [x for x in s["anyOf"] if x.get("type") != "null"]
        has_null  = any(x.get("type") == "null" for x in s["anyOf"])
        s = {**s, **(non_null[0] if non_null else {})}
        s.pop("anyOf", None)
        if has_null and not s.get("nullable"):
            s["nullable"] = True
    allowed = {"type", "description", "enum", "items", "properties", "required", "default", "nullable"}
    result = {k: v for k, v in s.items() if k in allowed}
    if "default" in result:
        result.setdefault("nullable", True)
    if "properties" in result:
        result["properties"] = {k: _clean_schema(v) for k, v in result["properties"].items()}
    if "items" in result:
        result["items"] = _clean_schema(result["items"])
    return result


_orig_get_tool_json_schema = _sm_models.get_tool_json_schema

def _get_tool_json_schema_clean(tool):
    result = _orig_get_tool_json_schema(tool)
    for prop in result["function"]["parameters"]["properties"].values():
        prop.pop("nullable", None)
    return result

_sm_models.get_tool_json_schema = _get_tool_json_schema_clean


_TS_HINT = " Must include timezone: use format \'YYYY-MM-DDTHH:MM:SSZ\' (e.g. \'2022-01-01T00:00:00Z\')."

def _mcp_tools(client):
    """Return schema-normalised tools from an MCPClient."""
    tools = client.get_tools()
    for t in tools:
        t.inputs = {name: _clean_schema(schema) for name, schema in t.inputs.items()}
        for name, schema in t.inputs.items():
            if "timestamp" in name and schema.get("type") == "string":
                schema["description"] = schema.get("description", "").rstrip(".") + _TS_HINT
            if name == "max_results" and not schema.get("nullable"):
                # Make max_results optional and inject a safe default if the model omits it
                schema["nullable"] = True
                schema["description"] = (
                    (schema.get("description") or "Number of results.").rstrip(".")
                    + " Defaults to 5 if omitted."
                )
                _orig_fwd = t.forward
                def _fwd_with_default(*args, _o=_orig_fwd, **kw):
                    kw.setdefault("max_results", 5)
                    return _o(*args, **kw)
                t.forward = _fwd_with_default
    return tools

In [6]:
def make_mcp_agent(max_steps: int = 10) -> ToolCallingAgent:
    """
    Create a ToolCallingAgent with a fresh MCP session.
    Always create a new instance per agent.run() — MCP sessions don't survive restarts.
    """
    client = MCPClient({"url": f"{AQL_API_BASE}/mcp/"}, structured_output=False)
    return ToolCallingAgent(tools=_mcp_tools(client), model=model, max_steps=max_steps)


# Smoke-test: list discovered tools
_agent = make_mcp_agent()
print(f"MCP tools available: {[t.name for t in _agent.tools.values() if t.name != 'final_answer']}")

MCP tools available: ['search_serps', 'count_serps', 'suggest_serps_queries', 'count_unique_serps', 'top_archives', 'top_providers', 'serp_date_histogram', 'compare_serps', 'get_serp_by_id', 'search_providers', 'count_providers', 'suggest_providers_queries', 'get_provider_by_id', 'search_archives', 'count_archives', 'suggest_archives_queries', 'get_archive_by_id']


In [ ]:
make_mcp_agent().run(
    "When did 'ChatGPT' first appear in AQL search queries, "
    "and how did search volume develop through 2023?"
)

In [ ]:
make_mcp_agent().run(
    "Find 5 example queries about 'COVID-19' in AQL. "
    "Which providers captured them and what years are they from?"
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find 5 example queries about 'COVID-19' in AQL. Which providers captured them and what years are they from?     │
│                                                                                                                 │
╰─ OpenAIModel - qwen3-30b-a3b ───────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'suggest_serps_queries' with arguments: {'query': 'COVID-19', 'size': 5}                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"result":|"RosalynMoran/Covid-19: Covid-19.","RosalynMoran/Covid-19: 
Covid-19.","covid-19?q=covid-19","RosalynMoran/Covid-19: Covid-19.","covid 19 19"]}

[Step 1: Duration 75.88 seconds| Input tokens: 7,158 | Output tokens: 1,592]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
make_mcp_agent(max_steps=15).run(
    "Compare search interest in 'global warming' and 'climate change' over time in AQL. "
    "Show yearly query counts for both terms and describe when and why the terminology shifted. "
    "Identify the years with the sharpest changes and link them to real-world events "
    "(scientific reports, political milestones, extreme weather). "
    "Name the most frequent example queries from the peak years."
)

---
## Optional — Gradio Chatbot UI

Interactive chat interface using the MCP agent (all AQL tools available).

In [ ]:
from smolagents import GradioUI

GradioUI(make_mcp_agent(), reset_agent_memory=True).launch()

---
## Your Turn

Five tiers — pick the depth that suits you.

**Form groups of 4–5.** Pick one question to investigate together and prepare a short show & tell:
what question did you ask, which tools did the agent call, and one thing that surprised you.

### Tier 1 — Try it (15 min)

Open the Gradio UI and ask the agent each of these questions. Note which tool it calls each time.

1. *"When did 'COVID-19' first appear in AQL search queries, and how did search volume develop through 2020?"*
2. *"Compare search interest in 'machine learning' and 'deep learning' over the years — when did deep learning overtake machine learning?"*
3. *"Which search providers in AQL have the most queries on climate science topics?"*
4. One question of your own — about any research topic you are curious about.

**Reflection:** Did the agent ever give an answer that surprised you? Was it grounded in the data or did it feel like a hallucination?


### Tier 2 — Add a custom tool (30–45 min)

Write a new `@tool`, add it alongside the MCP tools, and verify the agent uses it.

**Example:** a Wikipedia summary tool that lets the agent explain *why* a query spiked in a given year.

In [ ]:
from smolagents import tool


@tool
def wikipedia_summary(topic: str) -> str:
    """
    Fetch the opening paragraph of the Wikipedia article for a topic.
    Use this to provide background context for AQL query trends.

    Args:
        topic: The subject to look up (e.g. 'Bitcoin', 'COVID-19 pandemic').
    """
    import requests
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + topic.replace(" ", "_")
    r = requests.get(url, timeout=10, headers={"User-Agent": "AQLWorkshop/1.0 (educational)"})
    if not r.ok:
        return f"Not found (HTTP {r.status_code})"
    return r.json().get("extract", "No extract available")


def make_mcp_agent_extended(max_steps: int = 12) -> ToolCallingAgent:
    client = MCPClient({"url": f"{AQL_API_BASE}/mcp/"}, structured_output=False)
    return ToolCallingAgent(
        tools=_mcp_tools(client) + [wikipedia_summary],  # <- add your tool here
        model=model,
        max_steps=max_steps,
    )


# Test your extended agent
make_mcp_agent_extended().run(
    "In the year with the most bitcoin queries in AQL, "
    "what was happening with bitcoin according to Wikipedia?"
)

**Other tool ideas for Tier 2:**
- A tool that fetches top news headlines for a date (e.g. via NewsAPI or a public RSS feed)
- A tool that reads a local CSV and returns a summary or row lookup
- A tool that queries Google Trends for a keyword
- A tool that returns archive availability stats from the Wayback Machine CDX API

Once your tool works, ask the agent a question that *requires* it — verify in the trace that it was actually called.

### Tier 3 — Connect a public MCP server (~20 min)

The [official MCP server registry](https://github.com/modelcontextprotocol/servers) lists
community-maintained servers covering dozens of data sources.

The example below connects **arxiv-mcp-server** — the agent can now **search academic papers
on arXiv**. Combined with AQL, it can answer questions that span search trends and research output:

> *"When did searches for ‘neural retrieval’ peak — and what papers were published on arXiv that year?"*

In [ ]:
# Verify arxiv-mcp-server is installed
import shutil
assert shutil.which('arxiv-mcp-server'), 'arxiv-mcp-server not found — re-run cell 1'
print('\u2713 arxiv-mcp-server ready')

In [ ]:
# arxiv-mcp-server is a Python MCP server from the community registry.
# It runs as a subprocess — MCPClient starts it automatically.
#
# Other Python MCP servers from the community registry (pip install + command):
#   wikipedia-mcp         → Wikipedia article search and summaries
#   mcp-server-fetch      → fetch any web page as plain text
#   mcp-server-time       → current time and timezone conversions
#   paper-search-mcp      → multi-source academic search (arXiv, PubMed, Semantic Scholar, …)
#   mcp-server-git        → query a local git repository

from mcp import StdioServerParameters

def make_arxiv_agent(max_steps: int = 15) -> ToolCallingAgent:
    """AQL tools + arXiv paper search."""
    aql_client   = MCPClient({"url": f"{AQL_API_BASE}/mcp/"}, structured_output=False)
    arxiv_client = MCPClient(
        StdioServerParameters(
            command="arxiv-mcp-server",
            args=["--storage-path", "/tmp/arxiv"],
        ),
        structured_output=False,
    )
    return ToolCallingAgent(
        tools=_mcp_tools(aql_client) + _mcp_tools(arxiv_client),
        model=model,
        max_steps=max_steps,
    )

make_arxiv_agent().run(
    "Find the year in AQL with the most queries about 'information retrieval'. "
    "Then search arXiv for papers published that year on information retrieval "
    "and name the three most relevant ones."
)

### Tier 4 — Multi-agent orchestration

Instead of one agent doing everything, assign **specialist agents** to specific tasks and let an
**orchestrator** delegate and synthesise.

smolagents supports this via `ManagedAgent`: each sub-agent is wrapped as a callable tool that the
orchestrator can invoke.

In [ ]:
from smolagents import DuckDuckGoSearchTool

# Specialist 1: AQL data analyst
aql_specialist = make_mcp_agent(max_steps=8)
aql_specialist.name        = "aql_analyst"
aql_specialist.description = "Queries the Archive Query Log for historical search trends and statistics."

# Specialist 2: web researcher
web_specialist = ToolCallingAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model,
    max_steps=5,
    name="web_researcher",
    description="Searches the web for current news and context about any topic.",
)

# Orchestrator delegates to specialists via managed_agents
orchestrator = ToolCallingAgent(
    tools=[],
    model=model,
    max_steps=10,
    managed_agents=[aql_specialist, web_specialist],
)

orchestrator.run(
    "Find the year with the highest number of 'climate change' queries in AQL. "
    "Then search the web for the most significant climate events of that year. "
    "Write a short paragraph connecting the search trend to real-world events."
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find the year with the highest number of 'climate change' queries in AQL. Then search the web for the most      │
│ significant climate events of that year. Write a short paragraph connecting the search trend to real-world      │
│ events.                                                                                                         │
│                                                                                                                 │
╰─ OpenAIModel - qwen3-30b-a3b ───────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Tier 5 — Expose your own data via MCP (optional)

With [FastMCP](https://github.com/jlowin/fastmcp) you can turn **any dataset** — relevance
judgements, a paper corpus, experiment results — into a tool agents can call exactly like AQL.

The agent can then reason across **your data and AQL simultaneously**:

> *"In the years when people searched most for 'neural retrieval', what does my corpus say was
> published on that topic?"*

Your dataset becomes queryable by any MCP-compatible agent or collaborator.

In [ ]:
# Step 1: install FastMCP
# !pip install fastmcp

# Step 2: create my_server.py — expose your own data as MCP tools.
#
# Example: a lookup table of TREC shared tasks by year.
# Replace this with your own dataset (CSV, database, relevance judgements, ...).
#
#   from fastmcp import FastMCP
#   mcp = FastMCP("My Research Data")
#
#   @mcp.tool()
#   def get_trec_task(year: int) -> str:
#       """Return the main TREC shared tasks that ran in a given year."""
#       tasks = {
#           2019: "Deep Learning Track (passage + document retrieval)",
#           2020: "Deep Learning Track, Health Misinformation Track",
#           2021: "Deep Learning Track, Clinical Trials Track",
#           2022: "Deep Learning Track, NeuCLIR Track",
#           2023: "Deep Learning Track, TREC-RAG Track",
#       }
#       return tasks.get(year, f"No TREC data recorded for {year}")
#
#   if __name__ == "__main__":
#       mcp.run(transport="sse", port=9000)

# Step 3: run it in a terminal:  python my_server.py

# Step 4: connect both MCP servers — AQL + your own
def make_dual_agent(max_steps: int = 15) -> ToolCallingAgent:
    aql_client = MCPClient({"url": f"{AQL_API_BASE}/mcp/"}, structured_output=False)
    my_client  = MCPClient({"url": "http://localhost:9000/mcp/"}, structured_output=False)
    return ToolCallingAgent(
        tools=_mcp_tools(aql_client) + _mcp_tools(my_client),
        model=model,
        max_steps=max_steps,
    )


# make_dual_agent().run(
#     "Which year had the highest number of 'passage retrieval' queries in AQL? "
#     "What TREC shared tasks were running that year?"
# )